# Insurance Claim Risk Scoring + Explainability

Features are aligned with the real `InsuranceClaim` + `InsuranceClaimOcrAudit` entities from `insurance-service`.

**Workflow:**
1. Load raw synthetic data
2. Clean and normalize
3. Engineer risk features
4. Train a classifier
5. Evaluate and explain
6. Export model artifact for the FastAPI service

In [1]:
import joblib
import numpy as np
import pandas as pd

from pathlib import Path
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split, RandomizedSearchCV, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, classification_report, confusion_matrix
from sklearn.inspection import permutation_importance

pd.set_option("display.max_columns", 200)
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

In [2]:
path_candidates = [
    Path("../data/claims_risk_raw.csv"),
    Path("data/claims_risk_raw.csv"),
    Path("services/claim-risk-model/data/claims_risk_raw.csv"),
]
RAW_PATH = next((p for p in path_candidates if p.exists()), None)
if RAW_PATH is None:
    raise FileNotFoundError("claims_risk_raw.csv not found")

raw_df = pd.read_csv(RAW_PATH)
print(f"Loaded {len(raw_df):,} rows from {RAW_PATH}")
raw_df.head()

Loaded 150,000 rows from ..\data\claims_risk_raw.csv


,claim_id,user_id,amount,reimbursement_amount,insurance_company,insurance_grade,file_count,claim_date,status,ocr_decision,ocr_mismatch_count,ocr_major_count,ocr_minor_count,ocr_submitted_amount,ocr_extracted_amount,user_total_claims,user_claims_30d,days_since_last_claim,user_rejected_claims,fraud_flag
0,48703,1074,778.57,677.42,AMI,3.7,0,2025-10-15,PARTIALLY_APPROVED,NaN,NaN,NaN,NaN,NaN,NaN,19.0,3.0,218.0,1.0,0
1,9515,3278,4050.13,2394.21,BH,3.7,8,2024-10-20,APPROVED,NaN,NaN,NaN,NaN,NaN,NaN,12.0,3.0,60.0,2.0,0
2,59223,2190,1582.82,1325.66,BH,3.0,4,2024-10-05,PARTIALLY_APPROVED,PASS,0.0,0.0,0.0,1582.82,1532.01,11.0,0.0,336.0,1.0,0
3,140283,4053,7875.73,5414.66,CNAM,1.5,3,2024-04-10,SUBMITTED,PASS,0.0,0.0,0.0,7875.73,7715.15,2.0,2.0,139.0,0.0,0
4,35635,134,8051.35,6198.07,AMI,2.7,6,2025-12-04,APPROVED,PASS,0.0,0.0,0.0,8051.35,7666.30,18.0,0.0,NaN,0.0,0


In [3]:
# Data quality scan
quality = pd.DataFrame({
    "dtype": raw_df.dtypes.astype(str),
    "missing": raw_df.isna().sum(),
    "missing_pct": (raw_df.isna().mean() * 100).round(2),
    "empty_str": (raw_df == "").sum(),
    "unique": raw_df.nunique(dropna=False),
})
quality

,dtype,missing,missing_pct,empty_str,unique
claim_id,int64,0,0.00,0,150000
user_id,int64,0,0.00,0,5000
amount,float64,3087,2.06,0,136576
reimbursement_amount,float64,3014,2.01,0,134273
insurance_company,str,0,0.00,0,10
insurance_grade,float64,0,0.00,0,41
file_count,int64,0,0.00,0,9
claim_date,str,0,0.00,0,821
status,str,0,0.00,0,7
ocr_decision,str,25398,16.93,0,4


## Data Cleaning

In [4]:
def clean(df: pd.DataFrame) -> pd.DataFrame:
    c = df.copy()

    # Replace empty strings with NaN for proper imputation
    c.replace("", np.nan, inplace=True)

    # Numeric coercions
    num_cols = [
        "amount", "reimbursement_amount", "file_count",
        "ocr_mismatch_count", "ocr_major_count", "ocr_minor_count",
        "ocr_submitted_amount", "ocr_extracted_amount",
        "user_total_claims", "user_claims_30d", "days_since_last_claim",
        "user_rejected_claims", "fraud_flag",
    ]
    for col in num_cols:
        c[col] = pd.to_numeric(c[col], errors="coerce")

    c["insurance_grade"] = pd.to_numeric(c["insurance_grade"], errors="coerce").clip(1, 5)

    # Fill missing OCR decision as "NONE" (not yet analyzed)
    c["ocr_decision"] = c["ocr_decision"].fillna("NONE")

    # Impute numeric columns with median
    for col in num_cols:
        if col != "fraud_flag":
            c[col] = c[col].fillna(c[col].median())

    c["fraud_flag"] = c["fraud_flag"].fillna(0).astype(int)

    # Drop duplicates on claim_id
    c = c.drop_duplicates(subset=["claim_id"], keep="first")

    # --- Engineered features ---
    # Amount discrepancy between claimed and OCR-extracted
    c["amount_discrepancy"] = np.where(
        c["ocr_extracted_amount"] > 0,
        (c["amount"] - c["ocr_extracted_amount"]).abs() / c["ocr_extracted_amount"],
        0.0,
    )

    # Reimbursement ratio
    c["reimbursement_ratio"] = c["reimbursement_amount"] / c["amount"].replace(0, np.nan)
    c["reimbursement_ratio"] = c["reimbursement_ratio"].fillna(1.0).clip(0, 2)

    # High claim flag (top 10%)
    c["high_amount_flag"] = (c["amount"] > c["amount"].quantile(0.90)).astype(int)

    # Frequent claimant flag
    c["frequent_claimant_flag"] = (c["user_claims_30d"] > 3).astype(int)

    # Has rejections flag
    c["has_rejections_flag"] = (c["user_rejected_claims"] > 0).astype(int)

    # OCR severity score: weighted sum of major + minor
    c["ocr_severity"] = c["ocr_major_count"] * 3 + c["ocr_minor_count"]

    return c


clean_df = clean(raw_df)
print(f"Rows after cleaning: {len(clean_df):,}")
print(f"\nFraud distribution:\n{clean_df['fraud_flag'].value_counts(normalize=True).rename('pct')}")
clean_df.head()

Rows after cleaning: 150,000

Fraud distribution:
fraud_flag
0    0.879487
1    0.120513
Name: pct, dtype: float64


,claim_id,user_id,amount,reimbursement_amount,insurance_company,insurance_grade,file_count,claim_date,status,ocr_decision,ocr_mismatch_count,ocr_major_count,ocr_minor_count,ocr_submitted_amount,ocr_extracted_amount,user_total_claims,user_claims_30d,days_since_last_claim,user_rejected_claims,fraud_flag,amount_discrepancy,reimbursement_ratio,high_amount_flag,frequent_claimant_flag,has_rejections_flag,ocr_severity
0,48703,1074,778.57,677.42,AMI,3.7,0,2025-10-15,PARTIALLY_APPROVED,NONE,0.0,0.0,0.0,5006.825,4887.555,19.0,3.0,218.0,1.0,0,0.840704,0.870082,0,0,1,0.0
1,9515,3278,4050.13,2394.21,BH,3.7,8,2024-10-20,APPROVED,NONE,0.0,0.0,0.0,5006.825,4887.555,12.0,3.0,60.0,2.0,0,0.171338,0.591144,0,0,1,0.0
2,59223,2190,1582.82,1325.66,BH,3.0,4,2024-10-05,PARTIALLY_APPROVED,PASS,0.0,0.0,0.0,1582.820,1532.010,11.0,0.0,336.0,1.0,0,0.033166,0.837530,0,0,1,0.0
3,140283,4053,7875.73,5414.66,CNAM,1.5,3,2024-04-10,SUBMITTED,PASS,0.0,0.0,0.0,7875.730,7715.150,2.0,2.0,139.0,0.0,0,0.020814,0.687512,0,0,0,0.0
4,35635,134,8051.35,6198.07,AMI,2.7,6,2025-12-04,APPROVED,PASS,0.0,0.0,0.0,8051.350,7666.300,18.0,0.0,160.0,0.0,0,0.050226,0.769817,0,0,0,0.0


## Model Training

In [5]:
categorical_features = ["insurance_company", "ocr_decision", "status"]

numeric_features = [
    "amount", "reimbursement_amount", "insurance_grade", "file_count",
    "ocr_mismatch_count", "ocr_major_count", "ocr_minor_count",
    "ocr_submitted_amount", "ocr_extracted_amount",
    "user_total_claims", "user_claims_30d", "days_since_last_claim",
    "user_rejected_claims",
    "amount_discrepancy", "reimbursement_ratio", "high_amount_flag",
    "frequent_claimant_flag", "has_rejections_flag", "ocr_severity",
]

feature_cols = categorical_features + numeric_features

X = clean_df[feature_cols].copy()
y = clean_df["fraud_flag"].astype(int)

preprocessor = ColumnTransformer(
    transformers=[
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("ohe", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
        ]), categorical_features),
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]), numeric_features),
    ]
)

base_model = RandomForestClassifier(
    n_estimators=350,
    max_depth=12,
    min_samples_leaf=4,
    class_weight="balanced",
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

pipeline = Pipeline([
    ("prep", preprocessor),
    ("model", base_model),
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y,
)

# More complex training: cross-validation + hyperparameter search
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
param_distributions = {
    "model__n_estimators": [300, 500, 800],
    "model__max_depth": [10, 14, 18, None],
    "model__min_samples_split": [2, 5, 10],
    "model__min_samples_leaf": [1, 2, 4, 6],
    "model__max_features": ["sqrt", "log2", 0.4, 0.6],
}

search = RandomizedSearchCV(
    estimator=pipeline,
    param_distributions=param_distributions,
    n_iter=12,
    scoring="roc_auc",
    cv=cv,
    n_jobs=-1,
    verbose=1,
    random_state=RANDOM_STATE,
)

search.fit(X_train, y_train)

pipeline = search.best_estimator_
print("Training complete.")
print(f"Train size: {len(X_train):,}  Test size: {len(X_test):,}")
print("Best CV ROC-AUC:", round(search.best_score_, 4))
print("Best params:", search.best_params_)

Fitting 3 folds for each of 12 candidates, totalling 36 fits
Training complete.
Train size: 120,000  Test size: 30,000
Best CV ROC-AUC: 1.0
Best params: {'model__n_estimators': 800, 'model__min_samples_split': 2, 'model__min_samples_leaf': 4, 'model__max_features': 0.4, 'model__max_depth': 18}


## Evaluation

In [6]:
proba_test = pipeline.predict_proba(X_test)[:, 1]
pred_test = (proba_test >= 0.5).astype(int)

print("ROC-AUC:", round(roc_auc_score(y_test, proba_test), 4))
print("\nClassification report:\n")
print(classification_report(y_test, pred_test, digits=4))
print("Confusion matrix:\n", confusion_matrix(y_test, pred_test))

ROC-AUC: 0.9997

Classification report:

              precision    recall  f1-score   support

           0     0.9995    0.9992    0.9994     26385
           1     0.9945    0.9967    0.9956      3615

    accuracy                         0.9989     30000
   macro avg     0.9970    0.9980    0.9975     30000
weighted avg     0.9989    0.9989    0.9989     30000

Confusion matrix:
 [[26365    20]
 [   12  3603]]


## Feature Importance + Explainability

In [7]:
perm = permutation_importance(
    pipeline, X_test, y_test,
    n_repeats=8, random_state=RANDOM_STATE, scoring="roc_auc", n_jobs=-1,
)

importance_df = pd.DataFrame({
    "feature": X_test.columns,
    "importance_mean": perm.importances_mean,
    "importance_std": perm.importances_std,
}).sort_values("importance_mean", ascending=False)

importance_df.head(15)

,feature,importance_mean,importance_std
15,user_rejected_claims,3.226409e-02,5.669298e-04
14,days_since_last_claim,1.192013e-02,4.981221e-04
12,user_total_claims,4.839454e-03,2.626325e-04
13,user_claims_30d,3.524186e-03,1.327914e-04
19,frequent_claimant_flag,1.768564e-04,1.017896e-04
20,has_rejections_flag,5.652154e-05,1.008099e-05
17,reimbursement_ratio,4.348053e-05,4.607828e-06
11,ocr_extracted_amount,3.720706e-05,5.300489e-05
3,amount,2.983012e-05,5.568760e-05
5,insurance_grade,7.601033e-08,2.353547e-07


In [8]:
def risk_band(score: float) -> str:
    if score >= 70:
        return "HIGH"
    if score >= 40:
        return "MEDIUM"
    return "LOW"


importance_weight = {
    row.feature: max(float(row.importance_mean), 0.0001)
    for row in importance_df.itertuples(index=False)
}


def top_reasons(row: pd.Series) -> list[str]:
    reasons = []
    w = importance_weight

    if row.get("ocr_severity", 0) >= 3:
        reasons.append(("Major OCR mismatches detected", w.get("ocr_severity", 0.1) * row["ocr_severity"]))
    if row.get("ocr_mismatch_count", 0) >= 2:
        reasons.append(("Multiple OCR mismatches", w.get("ocr_mismatch_count", 0.1) * row["ocr_mismatch_count"]))
    if row.get("amount_discrepancy", 0) > 0.10:
        reasons.append(("Claimed amount differs from OCR extraction", w.get("amount_discrepancy", 0.1) * row["amount_discrepancy"]))
    if row.get("user_claims_30d", 0) > 3:
        reasons.append(("High claim frequency (30d)", w.get("user_claims_30d", 0.1) * row["user_claims_30d"]))
    if row.get("user_rejected_claims", 0) > 2:
        reasons.append(("History of rejected claims", w.get("user_rejected_claims", 0.1) * row["user_rejected_claims"]))
    if row.get("days_since_last_claim", 999) < 3:
        reasons.append(("Very recent previous claim", w.get("days_since_last_claim", 0.1) * (5 - row["days_since_last_claim"])))
    if row.get("high_amount_flag", 0) == 1:
        reasons.append(("Unusually high claim amount", w.get("high_amount_flag", 0.1)))
    if row.get("reimbursement_ratio", 0) > 0.95:
        reasons.append(("Near-full reimbursement ratio", w.get("reimbursement_ratio", 0.1) * row["reimbursement_ratio"]))
    ocr_dec = row.get("ocr_decision", "NONE")
    if ocr_dec == "MAJOR_BLOCKED":
        reasons.append(("OCR flagged as MAJOR_BLOCKED", w.get("ocr_decision", 0.1) * 3))
    elif ocr_dec == "NONE":
        reasons.append(("No OCR analysis performed yet", w.get("ocr_decision", 0.05)))
    if row.get("file_count", 2) == 0:
        reasons.append(("No supporting documents attached", w.get("file_count", 0.1)))

    if not reasons:
        reasons = [("No strong risk anomalies detected", 0.0)]

    reasons.sort(key=lambda x: x[1], reverse=True)
    return [r[0] for r in reasons[:3]]


# Score original cleaned claims
scored = clean_df.copy()
scored["risk_probability"] = pipeline.predict_proba(scored[feature_cols])[:, 1]
scored["risk_score"] = (scored["risk_probability"] * 100).round(1)
scored["risk_band"] = scored["risk_score"].apply(risk_band)
scored["top_reasons"] = scored.apply(top_reasons, axis=1)

scored[[
    "claim_id", "user_id", "amount", "insurance_company",
    "risk_score", "risk_band", "top_reasons", "fraud_flag",
]].sort_values("risk_score", ascending=False).head(15)

,claim_id,user_id,amount,insurance_company,risk_score,risk_band,top_reasons,fraud_flag
149987,68930,287,8621.28,LLOYD,100.0,HIGH,"[History of rejected claims, High claim freque...",1
19929,66527,4865,4733.52,CARTE,100.0,HIGH,"[History of rejected claims, High claim freque...",1
19938,14675,3707,9842.75,LLOYD,100.0,HIGH,"[History of rejected claims, Very recent previ...",1
19942,111442,1579,4832.68,CARTE,100.0,HIGH,"[History of rejected claims, High claim freque...",1
19945,41241,4184,9309.04,STAR,100.0,HIGH,"[History of rejected claims, Very recent previ...",1
19949,29581,1301,359.26,LLOYD,100.0,HIGH,"[History of rejected claims, Claimed amount di...",1
19954,23260,2642,6653.57,STAR,100.0,HIGH,"[History of rejected claims, High claim freque...",1
19959,112567,3153,5082.20,MAE,100.0,HIGH,"[History of rejected claims, Major OCR mismatc...",1
19963,33068,1766,7502.23,BH,100.0,HIGH,"[History of rejected claims, Very recent previ...",1
19972,84068,552,9671.72,LLOYD,100.0,HIGH,"[Very recent previous claim, High claim freque...",1


## Export Artifacts

In [9]:
ARTIFACT_DIR = Path("../models")
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

scored_path = ARTIFACT_DIR / "claims_risk_scored_output.csv"
model_path = ARTIFACT_DIR / "insurance_claim_risk_model.joblib"
importance_path = ARTIFACT_DIR / "insurance_claim_feature_importance.csv"

scored.to_csv(scored_path, index=False)
importance_df.to_csv(importance_path, index=False)
joblib.dump({
    "pipeline": pipeline,
    "feature_columns": feature_cols,
    "categorical_features": categorical_features,
    "numeric_features": numeric_features,
    "risk_band_thresholds": {"LOW": [0, 40], "MEDIUM": [40, 70], "HIGH": [70, 100]},
}, model_path)

print("Saved:")
print(f"  {scored_path.resolve()}")
print(f"  {importance_path.resolve()}")
print(f"  {model_path.resolve()}")

Saved:
  C:\Users\ambro\healthcare-system\services\claim-risk-model\models\claims_risk_scored_output.csv
  C:\Users\ambro\healthcare-system\services\claim-risk-model\models\insurance_claim_feature_importance.csv
  C:\Users\ambro\healthcare-system\services\claim-risk-model\models\insurance_claim_risk_model.joblib


## Integration

- Start the FastAPI service: `uvicorn app.main:app --port 5123 --reload` from `services/claim-risk-model/`
- `POST /score` accepts fields from `InsuranceClaim` + `InsuranceClaimOcrAudit` — only `claim_id`, `user_id`, `amount`, `reimbursement_amount`, `insurance_company`, `insurance_grade` are required
- `insurance-service` calls this Python service over HTTP to get `risk_score`, `risk_band`, and `top_reasons`
- Store scores in DB for audit trail and model monitoring